# Capítulo 4: Importando e usando dados vetoriais — a biblioteca OGR

**Chris Holden (ceholden@gmail.com) - [https://github.com/ceholden](https://github.com/ceholden)**

-----

## Configuração do Colab e Preparação

Para rodar este capítulo no Google Colab, precisamos garantir que as bibliotecas e utilitários de linha de comando (`gdalinfo`, `gdal_rasterize`) estejam disponíveis:

In [ ]:
# Instala as bibliotecas (repetido para garantir o ambiente no Colab)
!pip install gdal numpy --quiet

## Introdução

A biblioteca **OGR** é uma biblioteca companheira do **GDAL** que lida com recursos de dados vetoriais, incluindo consultas de informação, conversões de arquivo, rasterização de feições de polígono, vetorização de feições raster e muito mais. Ela suporta formatos populares, incluindo *ESRI Shapefile*, *Keyhole Markup Language* (KML), *PostGIS* e *SpatiaLite*.

> **Nota sobre o `gdal.OpenEx` (para sua informação):** O tutorial menciona que a API estava à frente da versão 1.11.0 na época, demonstrando o uso de `gdal.OpenEx` para unificar as seções GDAL e OGR da biblioteca. **No Python 3 moderno, `ogr.Open()` e `gdal.Open()` ainda são amplamente utilizados, e o `gdal.OpenEx` é o método preferido em novos desenvolvimentos, mas o código original com `ogr.Open()` geralmente funciona perfeitamente para abrir *datasets* vetoriais.**

Neste capítulo, usaremos um *ESRI Shapefile* que contém dados de treinamento para a imagem de exemplo que temos trabalhado.

## Abrindo um *ESRI Shapefile*

Assim como o GDAL, o OGR abstrai os formatos de arquivo para que possamos usar o mesmo código para qualquer formato. Ele emprega o mesmo conceito de um objeto *dataset*, do qual podemos coletar informações:

In [ ]:
# Importa OGR (da suíte osgeo)
from osgeo import ogr

# Caminho simulado para o arquivo de exemplo
shapefile_path = '/content/training_data.shp'

# NOTE: No Colab, você precisa garantir que o arquivo 'training_data.shp' e seus
# arquivos auxiliares (.shx, .dbf, .prj) foram carregados para o ambiente.
# Este código assume que o arquivo está em /content/.

# Abre o dataset a partir do arquivo
dataset = ogr.Open(shapefile_path)

# Verifica se o dataset existe -- seria None se não pudesse ser aberto
if not dataset:
    # Se você está executando no Colab e o arquivo não foi carregado,
    # esta mensagem de erro é esperada.
    print('Erro: não foi possível abrir o dataset. Certifique-se de carregar o Shapefile.')
else:
    print(f'Dataset "{shapefile_path}" aberto com sucesso.')

Com nosso Shapefile lido, podemos analisar algumas de suas propriedades:

In [ ]:
if dataset:
    # Obtemos o driver deste arquivo
    driver = dataset.GetDriver()
    print('Driver do Dataset é: {n}\n'.format(n=driver.name))

    # Quantas camadas (layers) estão contidas neste Shapefile?
    layer_count = dataset.GetLayerCount()
    print('O Shapefile tem {n} camada(s)\n'.format(n=layer_count))

    # Qual é o nome da 1ª camada? (Índice 0)
    layer = dataset.GetLayerByIndex(0)
    print('O nome da camada é: {n}\n'.format(n=layer.GetName()))

    # Qual é a geometria da camada? É um ponto? uma polilinha? um polígono?
    # Primeiro lemos a geometria - este é o valor do tipo enumerado
    geometry = layer.GetGeomType()

    # Em seguida, precisamos traduzi-lo para o nome do enum
    geometry_name = ogr.GeometryTypeToName(geometry)
    print("A geometria da camada é: {geom}\n".format(geom=geometry_name))

    # Qual é a projeção da camada?
    # Obtemos a referência espacial
    spatial_ref = layer.GetSpatialRef()

    # Exportamos esta referência espacial para algo que possamos ler... como o Proj4
    proj4 = spatial_ref.ExportToProj4()
    print('A projeção da camada é: {proj4}\n'.format(proj4=proj4))

    # Quantas feições (features) existem na camada?
    feature_count = layer.GetFeatureCount()
    print('A camada tem {n} feições\n'.format(n=feature_count))

    # Quantos campos existem no Shapefile e quais são seus nomes?
    # Primeiro, precisamos capturar a definição da camada
    defn = layer.GetLayerDefn()

    # Quantos campos
    field_count = defn.GetFieldCount()
    print('A camada tem {n} campos'.format(n=field_count))

    # Quais são os nomes?
    print('Seus nomes são: ')
    for i in range(field_count):
        field_defn = defn.GetFieldDefn(i)
        print('\t{name} - {datatype}'.format(name=field_defn.GetName(),
                                             datatype=field_defn.GetTypeName()))

O Shapefile já está projetado na mesma projeção que nossa imagem raster de exemplo. Se fosse necessário, você poderia reprojetar usando o utilitário de linha de comando [ogr2ogr](http://www.gdal.org/ogr2ogr.html) ou [reprojetando o shapefile em Python](http://pcjericks.github.io/py-gdalogr-cookbook/projection.html#reproject-a-layer).

## Relação com nosso *Dataset* Raster

Os dados de treinamento que acabamos de abrir contêm dois campos:

  * Um campo **ID** (tipo de dado Inteiro)
  * Um campo **class** (tipo de dado String)

Combinados com as informações de localização inatas dos polígonos em um Shapefile, os campos são tudo o que precisamos para emparelhar rótulos (ou seja, o ID inteiro e a descrição da string) com as informações em nosso raster.

Para emparelhar nossos dados vetoriais com os pixels raster, precisamos alinhar espacialmente os *datasets*. A maneira menos complicada de fazer isso é usar o conceito de uma imagem de Região de Interesse (ROI), onde cada valor de pixel diferente de zero em nossa imagem ROI corresponde a uma representação raster de um polígono de nossa camada vetorial.

Neste exemplo, o autor atribuiu valores variando de 1 a 5 para as classes:

  * **1** - floresta
  * **2** - água
  * **3** - herbáceas
  * **4** - área não vegetada (*barren*)
  * **5** - urbana

Para rasterizar uma camada vetorial, podemos usar a função GDAL `gdal.RasterizeLayer`, que é a abordagem *pure Python*, ou a utilidade de linha de comando `gdal_rasterize`.

### Versão Linha de Comando — `gdal_rasterize`

Primeiro, precisamos descobrir a extensão espacial e o tamanho do pixel de nosso raster de saída. Usaremos o `gdalinfo` para obter as informações da imagem original:

In [ ]:
%%bash
# Imprime metadados sobre o raster original (substitua o caminho conforme necessário)
# O `gdalinfo` no Colab pode ser chamado diretamente.
gdalinfo -proj4 ../../example/LE70220491999322EDC01_stack.gtif

A saída (simulada do original) nos fornece a informação necessária:

  * **Coordenadas Superior Esquerda (Upper Left)**: (462405.000, 1741815.000)
  * **Coordenadas Inferior Direita (Lower Right)**: (469905.000, 1734315.000)
  * **Tamanho do Pixel (Pixel Size)**: (30.000, -30.000)
  * **Projeção Proj.4**: `'+proj=utm +zone=15 +datum=WGS84 +units=m +no_defs '`

Usaremos essas informações para o comando `gdal_rasterize`. A ordem correta do switch `-te` é **`xmin ymin xmax ymax`**:

```bash
%%bash

# Caminhos simulados para o Colab.
input_shp='/content/training_data.shp'
output_tif='/content/training_data.gtif'
proj_str='+proj=utm +zone=15 +datum=WGS84 +units=m +no_defs'

# Explicação dos switches:
# -a "id" ==> grava valores do atributo "id" do shapefile
# -l training_data ==> o nome da camada do nosso shapefile
# -of "GTiff" ==> formato do arquivo raster de saída
# -a_srs ==> sistema de referência espacial de saída
# -a_nodata 0 ==> valor NODATA para o raster de saída (valor 0)
# -te ==> extensão alvo que corresponde ao raster para o qual queremos criar a imagem ROI
# -tr ==> resolução alvo, 30 x 30m
# -ot Byte ==> Como temos apenas valores de 0 a 5, um tipo de dado Byte é suficiente

# O comando Bash é mais fácil de programar em muitos aspectos
gdal_rasterize -a "id" \
    -l training_data \
    -of "GTiff" \
    -a_srs "${proj_str}" \
    -a_nodata 0 \
    -te 462405 1734315 469905 1741815 \
    -tr 30 30 \
    -ot Byte \
    "${input_shp}" "${output_tif}"
```

### Versão *Pure Python* — `gdal.RasterizeLayer`

Continuaremos com o método *pure Python* usando a função `gdal.RasterizeLayer`.

In [ ]:
# Importa GDAL (da suíte osgeo)
from osgeo import gdal
from osgeo import ogr # OGR já está importado, mas reforçamos

# Caminhos simulados para o Colab.
raster_file_path = '/content/LE70220491999322EDC01_stack.gtif'
output_tif_path = '/content/training_data.gtif'

# 1. Abrir o raster de exemplo para obter a projeção e a extensão
# NOTA: Este arquivo raster_file_path também deve ser carregado para o Colab
try:
    raster_ds = gdal.Open(raster_file_path, gdal.GA_ReadOnly)

    if raster_ds is None:
        # Se o raster não puder ser aberto, cria dados simulados para as variáveis de cabeçalho
        print(f"Raster original não encontrado. Criando metadados simulados.")
        ncol, nrow = 250, 250
        proj = '+proj=utm +zone=15 +datum=WGS84 +units=m +no_defs '
        # [ULx, x-res, x-rot, ULy, y-rot, y-res]
        ext = (462405.0, 30.0, 0.0, 1741815.0, 0.0, -30.0)
    else:
        # Busca o número de linhas e colunas
        ncol = raster_ds.RasterXSize
        nrow = raster_ds.RasterYSize

        # Busca a projeção e a extensão (GeoTransform)
        proj = raster_ds.GetProjectionRef()
        ext = raster_ds.GetGeoTransform()

        raster_ds = None # Fecha o dataset original
        print("Metadados do raster original capturados.")

except Exception as e:
    print(f"Erro ao acessar o raster original: {e}. Usando metadados simulados.")
    ncol, nrow = 250, 250
    proj = '+proj=utm +zone=15 +datum=WGS84 +units=m +no_defs '
    ext = (462405.0, 30.0, 0.0, 1741815.0, 0.0, -30.0)


# 2. Criar o dataset raster de saída (ROI)
# Usaremos o driver GTiff para o arquivo .gtif
memory_driver = gdal.GetDriverByName('GTiff')
out_raster_ds = memory_driver.Create(output_tif_path, ncol, nrow, 1, gdal.GDT_Byte) # 1 banda, tipo Byte

# 3. Definir a projeção e a extensão (GeoTransform) da imagem ROI
out_raster_ds.SetProjection(proj) #
out_raster_ds.SetGeoTransform(ext) #

# 4. Preencher a banda de saída com o valor 0 (NODATA / fundo)
b = out_raster_ds.GetRasterBand(1)
b.Fill(0)

# 5. Rasterizar a camada do shapefile para o nosso novo dataset
# A variável 'layer' é do código acima, assumindo que foi aberta com sucesso.
if 'layer' in locals() and layer is not None and layer.GetGeomType() != 0:
    status = gdal.RasterizeLayer(out_raster_ds,      # Saída para nosso novo dataset
                                 [1],                 # Saída para a primeira banda do nosso novo dataset
                                 layer,               # Rasteriza esta camada
                                 None, None,          # Não se preocupe com transformações, pois estamos na mesma projeção
                                 [0],                 # Burn value 0 (valor constante - não é usado com ATTRIBUTE)
                                 ['ALL_TOUCHED=TRUE',  # Rasteriza todos os pixels tocados por polígonos
                                  'ATTRIBUTE=id']     # Coloca valores raster de acordo com os valores do campo 'id'
                                 )
    # 6. Fechar dataset
    out_raster_ds = None # Fecha o raster de saída (salva no disco no caso do GTiff)

    if status != 0:
        print("A rasterização falhou...")
    else:
        print("Sucesso")
else:
    print("Rasterização não realizada: A camada vetorial não foi carregada com sucesso.")

## Verificação da Camada Rasterizada

Agora que temos um método funcional, podemos verificar quantos pixels de dados de treinamento coletamos:

In [ ]:
# Importa NumPy para algumas estatísticas
import numpy as np

# Abre o dataset rasterizado
try:
    roi_ds = gdal.Open(output_tif_path, gdal.GA_ReadOnly)

    if roi_ds is None:
        raise FileNotFoundError("Não foi possível abrir o arquivo rasterizado.")

    roi = roi_ds.GetRasterBand(1).ReadAsArray()

    # Quantos pixels estão em cada classe?
    classes = np.unique(roi)

    # Itera sobre todos os rótulos de classe na imagem ROI, imprimindo algumas informações
    for c in classes:
        # A classe 0 é a área NODATA/não classificada
        class_name = f"Classe {int(c)}" if int(c) > 0 else "Classe 0 (NODATA/Fundo)"
        print('{cls} contém {n} pixels'.format(cls=class_name,
                                                     n=(roi == c).sum()))

    roi_ds = None # Fecha o dataset

except Exception as e:
    print(f"Erro ao verificar a camada rasterizada: {e}")

## Conclusão

Agora que temos nossa imagem ROI, podemos usá-la para emparelhar nossos polígonos rotulados com os pixels correspondentes em nossa imagem Landsat para treinar um classificador para classificação de imagens. Continuamos esta etapa no próximo capítulo.